(genai-02-mm-llm)=
# Model monitoring using LLM

This tutorial illustrates a model monitoring system that leverages LLMs to maintain high standards for deployed models.

**In this tutorial**
- [Prerequisites](#prerequisites)
- [Add the monitoring-function code](#add-the-monitoring-function-code)
- [Deploy the model, enable tracking, and deploy the function](#deploy-the-model-enable-tracking-and-deploy-the-function)

This tutorial explains how an LLM can be monitored. To see it in action, run the [Large Language Model Monitoring](https://github.com/mlrun/demo-monitoring-and-feedback-loop/blob/main/README.md) demo.

## Prerequisites
- GPU node with NVIDIA drivers is necessary for the serving function

In [1]:
import mlrun
from mlrun.features import Feature
from mlrun.datastore.datastore_profile import DatastoreProfileV3io

Create the project

In [3]:
project = mlrun.get_or_create_project("genai-tutorial", "./", user_project=True)
project.set_source(".", pull_at_runtime=True)

> 2025-07-21 18:05:39,290 [info] Project loaded successfully: {"project_name":"genai-tutorial-xingsheng"}


Set the credentials

In [4]:
tsdb_profile = DatastoreProfileV3io(name="v3io-tsdb-profile")
project.register_datastore_profile(tsdb_profile)

stream_profile = DatastoreProfileV3io(
    name="v3io-stream-profile",
    v3io_access_key=mlrun.mlconf.get_v3io_access_key(),
)
project.register_datastore_profile(stream_profile)

In [5]:
project.set_model_monitoring_credentials(
    tsdb_profile_name=tsdb_profile.name,
    stream_profile_name=stream_profile.name,
)

Enable model monitoring for the project

In [6]:
project.enable_model_monitoring(
    base_period=2,  # frequency (in minutes) at which the monitoring applications are triggered
)

> 2025-07-21 18:05:44,543 [warning] enable_model_monitoring: 'base_period' < 10 minutes is not supported in production environments: {"project":"genai-tutorial-xingsheng"}


## Add the monitoring-function code

The monitoring function code collects the traffic to the serving function, analyzes it, and generates results for the specified metric.

In [7]:
%%writefile monit-code.py
import re
from typing import Any, Union

import mlrun
import mlrun.common.schemas
from mlrun.model_monitoring.applications import (
    ModelMonitoringApplicationBase,
    ModelMonitoringApplicationResult,
)

STATUS_RESULT_MAPPING = {
    0: mlrun.common.schemas.model_monitoring.constants.ResultStatusApp.detected,
    1: mlrun.common.schemas.model_monitoring.constants.ResultStatusApp.no_detection,
}


class LLMMonitoringFunction(ModelMonitoringApplicationBase):

    def do_tracking(
        self,
        monitoring_context,
    ) -> Union[
        ModelMonitoringApplicationResult, list[ModelMonitoringApplicationResult]
    ]:
        
        # User monitoring sampling, in this case an integer representing model performance
        # Can be calulated based off the traffic to the function using monitoring_context.sample_df
        result = 0.9

        monitoring_context.log_dataset(
            key="llm-monitoring-df",
            df=monitoring_context.sample_df
        )

        # get status:
        status = STATUS_RESULT_MAPPING[round(result)]

        return ModelMonitoringApplicationResult(
            name="llm_monitoring_df",
            value=result,
            kind=mlrun.common.schemas.model_monitoring.constants.ResultKindApp.model_performance,
            status=status,
            extra_data={},
        )

Writing monit-code.py


Define the model monitoring custom function that scans the traffic and calculates the performance metrics

In [8]:
application = project.set_model_monitoring_function(
    func="monit-code.py",
    application_class="LLMMonitoringFunction",
    name="llm-monit",
    image="mlrun/mlrun",
)

In [9]:
application.spec.readiness_timeout = 1200

In [10]:
project.deploy_function(application)

> 2025-07-21 18:06:03,905 [info] Starting remote function deploy
2025-07-21 18:06:04  (info) Deploying function
2025-07-21 18:06:04  (info) Building
2025-07-21 18:06:04  (info) Staging files and preparing base images
2025-07-21 18:06:04  (warn) Using user provided base image, runtime interpreter version is provided by the base image
2025-07-21 18:06:04  (info) Building processor image
2025-07-21 18:10:09  (info) Build complete
2025-07-21 18:10:20  (info) Function deploy complete
> 2025-07-21 18:10:27,983 [info] Model endpoint creation task completed with state succeeded
> 2025-07-21 18:10:27,983 [info] Successfully deployed function: {"external_invocation_urls":[],"internal_invocation_urls":["nuclio-genai-tutorial-xingsheng-llm-monit.default-tenant.svc.cluster.local:8080"]}


DeployStatus(state=ready, outputs={'endpoint': 'http://nuclio-genai-tutorial-xingsheng-llm-monit.default-tenant.svc.cluster.local:8080', 'name': 'genai-tutorial-xingsheng-llm-monit'})

Create a model serving class that loads the LLM and generates responses

In [11]:
%%writefile model-serving.py
import mlrun
from mlrun.serving.v2_serving import V2ModelServer
from transformers import AutoModelForCausalLM, AutoTokenizer
from typing import Any

class LLMModelServer(V2ModelServer):

    def __init__(
        self,
        context: mlrun.MLClientCtx = None,
        name: str = None,
        model_path: str = None,
        model_name: str = None,
        **kwargs
    ):
        super().__init__(name=name, context=context, model_path=model_path, **kwargs)
        self.model_name = model_name
    
    def load(
        self,
    ):
        # Load the model from Hugging Face
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForCausalLM.from_pretrained(self.model_name)


    def predict(self, request: dict[str, Any]):
        inputs = request.get("inputs", [])
      
        input_ids, attention_mask = self.tokenizer(
            inputs[0], return_tensors="pt"
        ).values()

        outputs = self.model.generate(input_ids=input_ids, attention_mask=attention_mask)

        # Remove input:
        outputs = self.tokenizer.decode(outputs[0])
        outputs = outputs.split(inputs[0])[-1].replace(self.tokenizer.eos_token, "")
        return [{"generated_text": outputs}]

Writing model-serving.py


Build an image

In [12]:
commands = [
    "pip install pytorch-lightning",
    "pip install packaging==21.3",
    "pip install transformers adapters openai",
]

In [13]:
# need different version of protobuf to work with different python version, python 3.9 -> protobuf 3.20.2, python 3.11 -> protobuf latest
import sys

minor_version = float(sys.version_info[1])
if minor_version >= 11:
    commands.append("pip install --upgrade --force-reinstall protobuf")
elif minor_version == 9:
    commands.append("pip install protobuf==3.20.2")
else:
    print(f"minor_version {minor_version} not supported")

Run the following command to build the image, once it's successfully built, no need to run the cell again since it takes quite sometime to build an image

In [ ]:
project.build_image(
    image=".llm-serving-base",
    base_image="mlrun/mlrun",
    set_as_default=False,
    commands=commands,
)

Create the serving function using the class you just defined

In [15]:
serving_fn = project.set_function(
    func="model-serving.py",
    name="llm-server",
    kind="serving",
    image=".llm-serving-base",
)

# Set readiness timeout to 20 minutes, deploy might take a while.
serving_fn.spec.readiness_timeout = 1200

# Attach fuse mount to the function
serving_fn.apply(mlrun.auto_mount())

## Deploy the model, enable tracking, and deploy the function

This tutorial uses the gpt2 model by Google. 

Log the model to the project

In [16]:
base_model = "gpt2"
project.log_model(
    base_model,
    model_file="src/model-iris.pkl",
    inputs=[Feature(value_type="str", name="question")],
    outputs=[Feature(value_type="str", name="answer")],
)

Adding the model parameters to the endpoint. This allow the model server class to initialize.

In [17]:
serving_fn.add_model(
    "gpt2",
    class_name="LLMModelServer",
    model_path=f"store://models/{project.name}/gpt2:latest",
    model_name="gpt2",
)

Enable tracking for the function, then deploy it.

In [18]:
serving_fn.set_tracking()

In [19]:
deployment = serving_fn.deploy()

> 2025-07-21 18:23:31,152 [info] Starting remote function deploy
2025-07-21 18:23:31  (info) Deploying function
2025-07-21 18:23:31  (info) Building
2025-07-21 18:23:31  (info) Staging files and preparing base images
2025-07-21 18:23:31  (warn) Using user provided base image, runtime interpreter version is provided by the base image
2025-07-21 18:23:31  (info) Building processor image
2025-07-21 18:25:51  (info) Build complete
2025-07-21 18:26:53  (info) Function deploy complete
> 2025-07-21 18:27:03,119 [info] Model endpoint creation task completed with state succeeded
> 2025-07-21 18:27:03,120 [info] Successfully deployed function: {"external_invocation_urls":["genai-tutorial-xingsheng-llm-server.default-tenant.app.cst-360.iguazio-cd0.com/"],"internal_invocation_urls":["nuclio-genai-tutorial-xingsheng-llm-server.default-tenant.svc.cluster.local:8080"]}


In [20]:
ret = serving_fn.invoke(
    path=f"/v2/models/{base_model}/infer",
    body={"inputs": ["What is a mortgage?"]},
)
ret

{'id': '124924cc-6cfd-49eb-8dda-f4a41dcbd14c',
 'model_name': 'gpt2',
 'outputs': [{'generated_text': '\n\nA mortgage is a loan that is made by a person who is not a resident of the'}],
 'timestamp': '2025-07-21 18:27:03.226350+00:00',
 'model_endpoint_uid': 'aefa9b699117497d9f164066f9aed17d'}

Test your model serving

Let's generate traffic against the model:

In [21]:
import time


def question_model(questions, serving_function, base_model):
    for question in questions:
        seconds = 0.5
        # Invoking the pretrained model:
        ret = serving_fn.invoke(
            path=f"/v2/models/{base_model}/infer",
            body={"inputs": [question]},
        )
        print(ret)
        time.sleep(seconds)

In [22]:
example_questions = [
    "What is a mortgage?",
    "How does a credit card work?",
    "Who painted the Mona Lisa?",
    "Please plan me a 4-days trip to north Italy",
    "Write me a song",
    "How much people are there in the world?",
    "What is climate change?",
    "How does the stock market work?",
    "Who wrote 'To Kill a Mockingbird'?",
    "Please plan me a 3-day trip to Paris",
    "Write me a poem about the ocean",
    "How many continents are there in the world?",
    "What is artificial intelligence?",
    "How does a hybrid car work?",
    "Who invented the telephone?",
    "Please plan me a week-long trip to New Zealand",
    "What is inflation?",
    "How do vaccines work?",
    "Who discovered gravity?",
    "Please plan me a weekend trip to Tokyo.",
    "Write me a short story about a time traveler.",
    "How many planets are in the solar system?",
    "What is quantum physics?",
    "How does a dishwasher work?",
    "Who wrote '1984'?",
    "Please plan me a 5-day road trip through California.",
    "Write me a haiku about autumn.",
    "What is the tallest mountain in the world?",
    "How does cryptocurrency work?",
    "Who invented the light bulb?",
    "What is the meaning of photosynthesis?",
    "How does an airplane fly?",
    "Who painted 'The Starry Night'?",
    "Please plan me a 10-day trip across South America.",
    "Write me a letter to apologize to a friend.",
    "How many countries are there in the world?",
    "What is renewable energy?",
    "How does Wi-Fi work?",
    "Who directed the movie 'Inception'?",
    "Please plan me a cultural tour of Egypt.",
]

In [23]:
question_model(
    questions=example_questions,
    serving_function=serving_fn,
    base_model=base_model,
)

{'id': 'c30f3e13-1104-4f08-8c58-58c02048ec2c', 'model_name': 'gpt2', 'outputs': [{'generated_text': '\n\nA mortgage is a loan that is made by a person who is not a resident of the'}], 'timestamp': '2025-07-21 18:30:13.713745+00:00', 'model_endpoint_uid': 'aefa9b699117497d9f164066f9aed17d'}
{'id': '2eef36c8-a43f-4e25-b3e1-bad9a9f2ab2f', 'model_name': 'gpt2', 'outputs': [{'generated_text': '\n\nA credit card is a payment card that is issued by a bank or other financial institution.'}], 'timestamp': '2025-07-21 18:30:16.120223+00:00', 'model_endpoint_uid': 'aefa9b699117497d9f164066f9aed17d'}
{'id': '981cb25d-80dc-4d4c-9184-e87885b6ccda', 'model_name': 'gpt2', 'outputs': [{'generated_text': '\n\nThe Mona Lisa is a very popular figure in the Marvel Universe. It was first seen'}], 'timestamp': '2025-07-21 18:30:18.411933+00:00', 'model_endpoint_uid': 'aefa9b699117497d9f164066f9aed17d'}
{'id': 'c0e1ac1a-070b-4202-9a77-2a73f18c5600', 'model_name': 'gpt2', 'outputs': [{'generated_text': ".\n\nI

Now the traffic to the function is analyzed and the performance is calculated.